# Week 2 — Probabilistic Fraud Detection Agent

## 1. Extending the Agent's State Space

In Week 1, the agent primarily estimated the probability
that a transaction was fraudulent.

In Week 2, the agent will explicitly reason about uncertainty.

Instead of only considering:

- Legitimate
- Fraudulent

the agent will consider four possible hidden states:

1. Legitimate & normal
2. Legitimate but unusual
3. Fraudulent
4. Unknown

The Unknown state represents situations where the available
evidence is insufficient to confidently determine the true state.

The goal is not to immediately classify every uncertain
transaction as fraud, but to allow the agent to decide whether
it should gather additional information before acting.

In [1]:
# Imports and data loading

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random
random.seed(42)

df_trans = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\fraud_transactions.csv")
df_cust = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\customer_profiles.csv")

df_trans["timestamp"] = pd.to_datetime(df_trans["timestamp"])

print("Transactions:", df_trans.shape)
print("Customers:", df_cust.shape)

Transactions: (12000, 8)
Customers: (1000, 5)


In [2]:
# Add customer profile information

df_trans = df_trans.merge(df_cust[["customer_id", "home_location"]],on="customer_id",how="left")

print(df_trans[["customer_id", "location", "home_location"]].head())

print("Customers without profile match:",df_trans["home_location"].isnull().sum())

  customer_id   location home_location
0       C0031       Pune          Pune
1       C0665        NaN     Hyderabad
2       C0422  Bengaluru     Bengaluru
3       C0107     Mumbai        Mumbai
4       C0455  Bengaluru     Bengaluru
Customers without profile match: 0


In [3]:
#SORT TRANSACTIONS CHRONOLOGICALLY

df_trans = df_trans.sort_values(["customer_id","timestamp"]).reset_index(drop=True)

print(df_trans[["transaction_id","customer_id","timestamp","amount"]].head(10))

# CREATE CUSTOMER HISTORIES

customer_histories = {customer_id: customer_df.copy() for customer_id, customer_df in df_trans.groupby("customer_id")}

print("Number of customers:",len(customer_histories))

  transaction_id customer_id           timestamp  amount
0         T00001       C0001 2026-08-01 15:19:00  799.20
1         T00002       C0001 2026-08-02 08:00:00  401.66
2         T00003       C0001 2026-08-03 15:34:00  516.92
3         T00004       C0001 2026-08-04 15:19:00  886.33
4         T00005       C0001 2026-08-05 17:14:00  795.07
5         T00006       C0001 2026-08-06 22:35:00  421.78
6         T00007       C0001 2026-08-07 19:41:00  406.29
7         T00008       C0001 2026-08-08 09:27:00  441.40
8         T00009       C0001 2026-08-09 19:34:00  480.65
9         T00010       C0001 2026-08-10 09:11:00  421.43
Number of customers: 1000


In [4]:
# Evidence Generation
#  GET CUSTOMER HISTORY

def get_customer_history(customer_id,timestamp,customer_histories):
    """
    Return only transactions that happened
    before the current transaction.
    """

    customer = customer_histories[customer_id]

    current_time = pd.to_datetime(timestamp)

    history = customer[customer["timestamp"] < current_time]

    return history

# AMOUNT UNUSUAL

def check_amount(current_amount,history):
    """
    Returns True when the current transaction
    is more than 4x the customer's historical
    average transaction amount.
    """

    if history.empty:
        return False

    average_amount = history["amount"].mean()

    if average_amount == 0:
        return False

    return (current_amount >4 * average_amount)

# LOCATION UNUSUAL

def check_location(current_location,historical_locations):
    """
    Returns:
    True  -> location has not been seen before
    False -> location has been seen before
    None  -> location is missing
    """

    if pd.isna(current_location):
        return None

    if current_location in historical_locations.values:
        return False

    return True

#  MERCHANT UNUSUAL

def check_merchant(current_merchant,historical_merchants):
    """
    Returns:
    True  -> merchant has not been seen before
    False -> merchant has been seen before
    None  -> merchant is missing
    """

    if pd.isna(current_merchant):
        return None

    if current_merchant in historical_merchants.values:
        return False

    return True

# FREQUENCY UNUSUAL

def check_frequency(current_timestamp,history):
    """
    Compare the current transaction gap
    with the customer's historical median gap.

    A transaction is unusual when the current gap
    is less than 50% of the normal median gap.
    """

    if history.empty:
        return False

    current_timestamp = pd.to_datetime(current_timestamp)

    history = history.sort_values("timestamp")

    previous_transaction = history.iloc[-1]["timestamp"]

    current_gap = (current_timestamp -previous_transaction)

    historical_gaps = (history["timestamp"].diff().dropna())

    if historical_gaps.empty:
        return False

    normal_median_gap = (historical_gaps.median())

    frequency_threshold = (normal_median_gap * 0.5)

    return (current_gap <frequency_threshold)

#  VELOCITY UNUSUAL

def check_velocity(current_timestamp,history):
    """
    Returns True when at least 3 previous
    transactions occurred within 30 minutes.
    """

    if history.empty:
        return False

    current_timestamp = pd.to_datetime(current_timestamp)

    time_difference_minutes = ((current_timestamp -history["timestamp"]).dt.total_seconds()/ 60)

    recent_transactions = history[(time_difference_minutes >= 0) & (time_difference_minutes <= 30)]

    return (len(recent_transactions) >= 3)

# GENERATE ALL EVIDENCE

def get_evidence(transaction,customer_histories):

    history = get_customer_history(transaction["customer_id"],transaction["timestamp"],customer_histories)

    amount_unusual = check_amount(transaction["amount"],history)

    location_unusual = check_location(transaction["location"],history["location"])
    
    merchant_unusual = check_merchant(transaction["merchant"],history["merchant"])

    frequency_unusual = check_frequency(transaction["timestamp"],history)

    velocity_unusual = check_velocity(transaction["timestamp"],history)

    return {
        "amount_unusual":
            amount_unusual,

        "location_unusual":
            location_unusual,

        "merchant_unusual":
            merchant_unusual,

        "frequency_unusual":
            frequency_unusual,

        "velocity_unusual":
            velocity_unusual
    }

# GENERATE EVIDENCE FOR ALL TRANSACTIONS

evidence_rows = []

for _, transaction in df_trans.iterrows():

    evidence = get_evidence(
        transaction,
        customer_histories
    )

    evidence_rows.append(evidence)

evidence_df = pd.DataFrame(
    evidence_rows
)

print("Evidence dataframe shape:", evidence_df.shape)

print("\nEvidence columns:")
print(evidence_df.columns.tolist())

print("\nFirst 5 rows:")
print(evidence_df.head())

Evidence dataframe shape: (12000, 5)

Evidence columns:
['amount_unusual', 'location_unusual', 'merchant_unusual', 'frequency_unusual', 'velocity_unusual']

First 5 rows:
   amount_unusual location_unusual merchant_unusual  frequency_unusual  \
0           False             True             True              False   
1           False            False             True              False   
2           False            False            False              False   
3           False            False            False              False   
4           False            False            False              False   

   velocity_unusual  
0             False  
1             False  
2             False  
3             False  
4             False  


In [5]:
#  ADD EVIDENCE TO TRANSACTION DATASET

evidence_columns = [
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual"
]

for column in evidence_columns:

    df_trans[column] = evidence_df[column].values

print(
    df_trans[
        [
            "transaction_id",
            "amount",
            "location",
            "merchant",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "fraud"
        ]
    ].head(10)
)

# EVIDENCE DISTRIBUTION
for column in evidence_columns:

    print("\n" + "=" * 60)
    print(column.upper())
    print("=" * 60)

    print(
        df_trans[column]
        .value_counts(dropna=False)
    )
    
#  EVIDENCE PERCENTAGES BY FRAUD CLASS

for column in evidence_columns:

    print("\n" + "=" * 60)
    print(column.upper())
    print("=" * 60)

    result = pd.crosstab(
        df_trans[column],
        df_trans["fraud"],
        normalize="columns"
    ) * 100

    print(
        result.round(2)
    )
# COUNT SUSPICIOUS SIGNALS

df_trans["suspicious_signal_count"] = (df_trans[evidence_columns].fillna(False).astype(int).sum(axis=1))

print(
    df_trans[
        "suspicious_signal_count"
    ].value_counts()
    .sort_index()
)

# ============================================================
# FINAL WEEK 2 DATASET
# ============================================================

week2_columns = [
    "transaction_id",
    "customer_id",
    "amount",
    "timestamp",
    "location",
    "merchant",
    "merchant_category",
    "home_location",

    # Week 2 observable evidence
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual",

    # Exploratory feature
    "suspicious_signal_count",

    # Actual label - used only for evaluation
    "fraud"
]

week2_df = df_trans[
    week2_columns
].copy()

print(
    "Week 2 dataset shape:",
    week2_df.shape
)

print(
    "\nWeek 2 columns:"
)

print(
    week2_df.columns.tolist()
)

print(
    "\nFirst 10 rows:"
)

print(
    week2_df.head(10)
)

  transaction_id  amount location   merchant  amount_unusual location_unusual  \
0         T00001  799.20   Mumbai     Myntra           False             True   
1         T00002  401.66   Mumbai        H&M           False            False   
2         T00003  516.92   Mumbai     Myntra           False            False   
3         T00004  886.33   Mumbai     Myntra           False            False   
4         T00005  795.07   Mumbai     Myntra           False            False   
5         T00006  421.78   Mumbai        H&M           False            False   
6         T00007  406.29   Mumbai     Myntra           False            False   
7         T00008  441.40   Mumbai     Myntra           False            False   
8         T00009  480.65   Mumbai  Big Chill           False            False   
9         T00010  421.43   Mumbai        H&M           False            False   

  merchant_unusual  frequency_unusual  velocity_unusual  fraud  
0             True              False      

C:\Users\user\AppData\Local\Temp\ipykernel_13272\593215378.py:63: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_trans["suspicious_signal_count"] = (df_trans[evidence_columns].fillna(False).astype(int).sum(axis=1))


In [6]:
# Calculate likelihoods
########## Among fraudlent transactions, how often X is unusual
def calculate_likelihoods(df_trans):
    fraud_trans = df_trans.loc[df_trans["fraud"] == 1]
    legit_trans = df_trans.loc[df_trans["fraud"] == 0]

    total_fraud = len(fraud_trans)
    total_legit = len(legit_trans)

    fraud_amount_count = (fraud_trans["amount_unusual"] == True).sum()
    legit_amount_count = (legit_trans["amount_unusual"] == True).sum()

    fraud_location_count = (fraud_trans["location_unusual"] == True).sum()
    legit_location_count = (legit_trans["location_unusual"] == True).sum()

    fraud_merchant_count = (fraud_trans["merchant_unusual"] == True).sum()
    legit_merchant_count = (legit_trans["merchant_unusual"] == True).sum()

    fraud_frequency_count = (fraud_trans["frequency_unusual"] == True).sum()
    legit_frequency_count = (legit_trans["frequency_unusual"] == True).sum()

    fraud_velocity_count = (fraud_trans["velocity_unusual"] == True).sum()
    legit_velocity_count = (legit_trans["velocity_unusual"] == True).sum()

    # Laplace smoothing
    p_amount_fraud = (fraud_amount_count + 1) / (total_fraud + 2)
    p_amount_legit = (legit_amount_count + 1) / (total_legit + 2)

    p_location_fraud = (fraud_location_count + 1) / (total_fraud + 2)
    p_location_legit = (legit_location_count + 1) / (total_legit + 2)

    p_merchant_fraud = (fraud_merchant_count + 1) / (total_fraud + 2)
    p_merchant_legit = (legit_merchant_count + 1) / (total_legit + 2)

    p_frequency_fraud = (fraud_frequency_count + 1) / (total_fraud + 2)
    p_frequency_legit = (legit_frequency_count + 1) / (total_legit + 2)

    p_velocity_fraud = (fraud_velocity_count + 1) / (total_fraud + 2)
    p_velocity_legit = (legit_velocity_count + 1) / (total_legit + 2)

    return (
        p_amount_fraud, p_amount_legit,
        p_location_fraud, p_location_legit,
        p_merchant_fraud, p_merchant_legit,
        p_frequency_fraud, p_frequency_legit,
        p_velocity_fraud, p_velocity_legit
    )


likelihoods = calculate_likelihoods(df_trans)
print(likelihoods)

(np.float64(0.2647352647352647), np.float64(9.998000399920016e-05), np.float64(0.35564435564435565), np.float64(0.10007998400319935), np.float64(0.20279720279720279), np.float64(0.33833233353329334), np.float64(0.9995004995004995), np.float64(0.028594281143771244), np.float64(0.0004995004995004995), np.float64(9.998000399920016e-05))


In [7]:
#  Calculate fraud belief

def calculate_fraud_belief(evidence, likelihoods):

    prior_fraud = 0.05
    prior_legit = 1 - prior_fraud

    (
        p_amount_fraud,
        p_amount_legit,
        p_location_fraud,
        p_location_legit,
        p_merchant_fraud,
        p_merchant_legit,
        p_frequency_fraud,
        p_frequency_legit,
        p_velocity_fraud,
        p_velocity_legit
    ) = likelihoods

    if evidence["amount_unusual"] == True:
        amount_fraud = p_amount_fraud  #If unusual amount is observed, use P(unusual amount | fraud).
        amount_legit = p_amount_legit  #If normal amount is observed, use P(normal amount | fraud).
    else:
        amount_fraud = 1 - p_amount_fraud
        amount_legit = 1 - p_amount_legit

    if evidence["location_unusual"] == True:
        location_fraud = p_location_fraud
        location_legit = p_location_legit
    elif evidence["location_unusual"] == False:
        location_fraud = 1 - p_location_fraud
        location_legit = 1 - p_location_legit
    else:
        location_fraud = 1
        location_legit = 1

    if evidence["merchant_unusual"] == True:
        merchant_fraud = p_merchant_fraud
        merchant_legit = p_merchant_legit
    elif evidence["merchant_unusual"] == False:
        merchant_fraud = 1 - p_merchant_fraud
        merchant_legit = 1 - p_merchant_legit
    else:
        merchant_fraud = 1
        merchant_legit = 1

    if evidence["frequency_unusual"] == True:
        frequency_fraud = p_frequency_fraud
        frequency_legit = p_frequency_legit
    else:
        frequency_fraud = 1 - p_frequency_fraud
        frequency_legit = 1 - p_frequency_legit

    if evidence["velocity_unusual"] == True:
        velocity_fraud = p_velocity_fraud
        velocity_legit = p_velocity_legit
    else:
        velocity_fraud = 1 - p_velocity_fraud
        velocity_legit = 1 - p_velocity_legit

    fraud_probability = (prior_fraud * amount_fraud * location_fraud * merchant_fraud * frequency_fraud * velocity_fraud)

    legit_probability = (prior_legit  * amount_legit * location_legit * merchant_legit * frequency_legit * velocity_legit)

    return fraud_probability / (fraud_probability + legit_probability)

### Cost Assumptions

The action costs used by the agent are simulated values for this experiment.

They represent the relative consequences of different decisions:

- Approving a fraudulent transaction has a high cost.
- Questioning a transaction has a lower cost because additional information is requested.
- Examining a transaction represents a review process with an associated cost.
- Declining a legitimate transaction has a cost because it may inconvenience the customer.

These values are not based on real banking data. In a production system, they would need to be estimated using actual business, customer, and operational costs.

In [8]:
#  Cost model and action selection

def calculate_action_costs(fraud_belief):
    legit_belief = 1 - fraud_belief

    approve_cost = fraud_belief * 100
    question_cost = (fraud_belief * 10 + legit_belief * 2)
    examine_cost = (fraud_belief * 5  + legit_belief * 8)
    decline_cost = legit_belief * 20

    return {
        "Approve": approve_cost,
        "Question": question_cost,
        "Examine": examine_cost,
        "Decline": decline_cost
    }


def choose_action(costs):
    return min(costs, key=costs.get)

In [9]:
print("Transactions:", df_trans.shape)
print("Evidence columns:")
print([
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual"
])

print("\nSample transaction:")
display(df_trans.head(1))

Transactions: (12000, 15)
Evidence columns:
['amount_unusual', 'location_unusual', 'merchant_unusual', 'frequency_unusual', 'velocity_unusual']

Sample transaction:


,transaction_id,customer_id,amount,timestamp,location,merchant,merchant_category,fraud,home_location,amount_unusual,location_unusual,merchant_unusual,frequency_unusual,velocity_unusual,suspicious_signal_count
0,T00001,C0001,799.2,2026-08-01 15:19:00,Mumbai,Myntra,E-commerce,0,Mumbai,False,True,True,False,False,2


## Train/Test Split

Week 1 calculated likelihoods using the available transaction data.

In Week 2, we separate the data into training and test sets.

The training data is used to estimate the likelihood of different evidence signals, while the test data is used to evaluate how the agent performs on unseen transactions.

In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_trans,test_size=0.20,random_state=42)

print("Training transactions:", train_df.shape)
print("Test transactions:", test_df.shape)

Training transactions: (9600, 15)
Test transactions: (2400, 15)


In [11]:
train_likelihoods = calculate_likelihoods(train_df)

print("Training likelihoods calculated successfully.")

Training likelihoods calculated successfully.


## Initial Fraud Belief

For each test transaction, the agent first observes the available evidence and calculates its current belief that the transaction is fraudulent.

This is the agent's starting point before requesting any additional information.

In [12]:
# Take one transaction from the test set
test_transaction = test_df.iloc[0]

# Extract its evidence
test_evidence = {"amount_unusual": test_transaction["amount_unusual"],
    "location_unusual": test_transaction["location_unusual"],
    "merchant_unusual": test_transaction["merchant_unusual"],
    "frequency_unusual": test_transaction["frequency_unusual"],
    "velocity_unusual": test_transaction["velocity_unusual"]
}

# Calculate initial fraud belief
initial_belief = calculate_fraud_belief(test_evidence,train_likelihoods)

print("Transaction:", test_transaction["transaction_id"])
print("Actual fraud label:", test_transaction["fraud"])
print("Initial fraud belief:", initial_belief)

Transaction: T01628
Actual fraud label: 0
Initial fraud belief: 1.7722808597665913e-05


## Inital cost-sensitive decision
The agent uses its current belief to esitmate the cost of every possible action and then selects the actions with the lowest expected cost.

In [13]:
initial_costs = calculate_action_costs(initial_belief)

initial_action = choose_action(initial_costs)

print("Initial fraud belief:", initial_belief)
print("\nAction costs:")

for action, cost in initial_costs.items():
    print(f"{action}: {cost:.4f}")

print("\nInitial action:", initial_action)

Initial fraud belief: 1.7722808597665913e-05

Action costs:
Approve: 0.0018
Question: 2.0001
Examine: 7.9999
Decline: 19.9996

Initial action: Approve


## Update Belief with New Evidence 
When the agent asks the customer for additional info, then whatever the customer will response will become the new evidence.
The agent will then update its fraud belief.

In [14]:
def update_belief_with_customer_response(fraud_belief,customer_response):
    legit_belief = 1 - fraud_belief

    # Assumed response likelihoods for simulation
    p_yes_fraud = 0.10
    p_yes_legit = 0.95

    if customer_response == True:
        p_response_fraud = p_yes_fraud
        p_response_legit = p_yes_legit

    elif customer_response == False:
        p_response_fraud = 1 - p_yes_fraud
        p_response_legit = 1 - p_yes_legit

    else:
        return fraud_belief

    fraud_probability = fraud_belief * p_response_fraud
    legit_probability = legit_belief * p_response_legit

    updated_belief = (
        fraud_probability /
        (fraud_probability + legit_probability)
    )

    return updated_belief

## Belief update after the customer has responded
The agent compares its belief before and after recieving the additional info.
A response that is more likely from a legitimate customer should reduce the fraud belief, whereas a response that is more likely a fraud should increase the fraud belief

In [15]:
# Test with customer confirming the transaction
updated_belief_yes = update_belief_with_customer_response(initial_belief,True)

# Test with customer denying the transaction
updated_belief_no = update_belief_with_customer_response(initial_belief,False)

print("Initial fraud belief:", initial_belief)
print("After customer says YES:", updated_belief_yes)
print("After customer says NO:", updated_belief_no)

Initial fraud belief: 1.7722808597665913e-05
After customer says YES: 1.865588382855314e-06
After customer says NO: 0.0003189144697361814


## Recalculating decision after the new evidence
After updating its fraud belief, the agent will recalculate the cost of each possible actions.
and hence the agent will change its decision based on the new information

In [16]:
# Recalculate costs after customer says NO
updated_costs_no = calculate_action_costs(updated_belief_no)

updated_action_no = choose_action(updated_costs_no)

print("Updated fraud belief:", updated_belief_no)

print("\nUpdated action costs:")
for action, cost in updated_costs_no.items():
    print(f"{action}: {cost:.4f}")

print("\nUpdated action:", updated_action_no)

Updated fraud belief: 0.0003189144697361814

Updated action costs:
Approve: 0.0319
Question: 2.0026
Examine: 7.9990
Decline: 19.9936

Updated action: Approve


## Selecting Additional Information
When the agent decides that more information is needed, it identifies which evidence is still uncertain and asks the customer for the most relevant information.
The goal is not to ask every possible question, but to gather information that can help reduce uncertainty.

In [17]:
def get_questions(evidence):

    questions = []

    if evidence["amount_unusual"]:
        questions.append("transaction_confirmation")

    if evidence["frequency_unusual"]:
        questions.append("transaction_confirmation")

    if evidence["location_unusual"]:
        questions.append("location_confirmation")

    return list(set(questions))

### Simulation Note

Customer responses are simulated using the actual fraud label to create realistic synthetic responses for this experiment.

The fraud label is **not provided to the agent** during decision-making. It is used only to simulate how a legitimate customer or fraudulent transaction might respond.

In a real-world system, these response probabilities would need to be learned from historical customer-response data rather than using the actual fraud label.

In [18]:
import random

def simulate_customer_response(transaction, questions):

    responses = {}

    for question in questions:

        if question == "transaction_confirmation":

            if transaction["fraud"] == 0:
                responses[question] = random.random() < 0.95
            else:
                responses[question] = random.random() < 0.10

        elif question == "location_confirmation":

            if transaction["fraud"] == 0:
                responses[question] = random.random() < 0.90
            else:
                responses[question] = random.random() < 0.20

    return responses

## Asking additional information
When the agent needs more info, it selects the relevant questions and receives a simulated customer response. The response is then later used to update the agent's belief.

In [19]:
# Use the test transaction from earlier

questions = get_questions(test_evidence)

print("Transaction:", test_transaction["transaction_id"])
print("Questions:", questions)

customer_responses = simulate_customer_response(
    test_transaction,
    questions
)

print("Customer responses:", customer_responses)

Transaction: T01628
Questions: []
Customer responses: {}


In [20]:
# Find test transactions where at least one evidence signal is unusual

evidence_columns = [
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual"
]

uncertain_test_transactions = test_df[
    test_df[evidence_columns].any(axis=1)
]

print("Transactions with at least one unusual signal:",
      len(uncertain_test_transactions))

display(
    uncertain_test_transactions[
        ["transaction_id", "fraud"] + evidence_columns
    ].head(5)
)

Transactions with at least one unusual signal: 1098


,transaction_id,fraud,amount_unusual,location_unusual,merchant_unusual,frequency_unusual,velocity_unusual
1720,T01442,0,False,False,True,False,False
9120,T07612,0,False,False,True,False,False
9663,AMT_C0806,1,True,False,False,True,False
4617,FREQ_C0386,1,False,False,False,True,False
1408,MERCH_C0118,1,False,True,True,True,False


In [21]:
# Select a transaction with multiple unusual signals
active_test_transaction = test_df[
    test_df["transaction_id"] == "AMT_C0806"
].iloc[0]

active_evidence = {
    "amount_unusual": active_test_transaction["amount_unusual"],
    "location_unusual": active_test_transaction["location_unusual"],
    "merchant_unusual": active_test_transaction["merchant_unusual"],
    "frequency_unusual": active_test_transaction["frequency_unusual"],
    "velocity_unusual": active_test_transaction["velocity_unusual"]
}

# Calculate initial belief
active_initial_belief = calculate_fraud_belief(
    active_evidence,
    train_likelihoods
)

# Calculate initial action
active_initial_costs = calculate_action_costs(
    active_initial_belief
)

active_initial_action = choose_action(
    active_initial_costs
)

print("Transaction:", active_test_transaction["transaction_id"])
print("Actual fraud:", active_test_transaction["fraud"])
print("Evidence:", active_evidence)
print("Initial fraud belief:", active_initial_belief)
print("Initial action:", active_initial_action)

Transaction: AMT_C0806
Actual fraud: 1
Evidence: {'amount_unusual': np.True_, 'location_unusual': False, 'merchant_unusual': False, 'frequency_unusual': np.True_, 'velocity_unusual': np.False_}
Initial fraud belief: 0.9997158073927304
Initial action: Decline


In [22]:
question_transactions = []

for _, transaction in test_df.iterrows():

    evidence = {
        "amount_unusual": transaction["amount_unusual"],
        "location_unusual": transaction["location_unusual"],
        "merchant_unusual": transaction["merchant_unusual"],
        "frequency_unusual": transaction["frequency_unusual"],
        "velocity_unusual": transaction["velocity_unusual"]
    }

    belief = calculate_fraud_belief(
        evidence,
        train_likelihoods
    )

    costs = calculate_action_costs(belief)
    action = choose_action(costs)

    if action == "Question":
        question_transactions.append(
            (transaction, evidence, belief, costs)
        )

print("Transactions with initial Question action:",len(question_transactions))

Transactions with initial Question action: 223


In [23]:
# Select the first transaction where the agent initially chooses Question

question_transaction, question_evidence, question_belief, question_costs = (
    question_transactions[0]
)

print("Transaction:", question_transaction["transaction_id"])
print("Actual fraud:", question_transaction["fraud"])
print("Evidence:", question_evidence)
print("Initial fraud belief:", question_belief)

print("\nInitial action costs:")
for action, cost in question_costs.items():
    print(f"{action}: {cost:.4f}")

print("\nInitial action: Question")

Transaction: FREQ_C0386
Actual fraud: 1
Evidence: {'amount_unusual': False, 'location_unusual': False, 'merchant_unusual': False, 'frequency_unusual': True, 'velocity_unusual': False}
Initial fraud belief: 0.5421293559204614

Initial action costs:
Approve: 54.2129
Question: 6.3370
Examine: 6.3736
Decline: 9.1574

Initial action: Question


In [24]:
questions = get_questions(question_evidence)

customer_responses = simulate_customer_response(
    question_transaction,
    questions
)

print("Questions asked:", questions)
print("Customer responses:", customer_responses)

customer_response = customer_responses["transaction_confirmation"]

updated_question_belief = update_belief_with_customer_response(
    question_belief,
    customer_response
)

print("Initial fraud belief:", question_belief)
print("Customer response:", customer_response)
print("Updated fraud belief:", updated_question_belief)

Questions asked: ['transaction_confirmation']
Customer responses: {'transaction_confirmation': False}
Initial fraud belief: 0.5421293559204614
Customer response: False
Updated fraud belief: 0.955181898687604


In [25]:
updated_question_costs = calculate_action_costs(updated_question_belief)

updated_question_action = choose_action(updated_question_costs)

print("Updated fraud belief:", updated_question_belief)

print("\nUpdated action costs:")
for action, cost in updated_question_costs.items():
    print(f"{action}: {cost:.4f}")

print("\nUpdated action:", updated_question_action)

Updated fraud belief: 0.955181898687604

Updated action costs:
Approve: 95.5182
Question: 9.6415
Examine: 5.1345
Decline: 0.8964

Updated action: Decline


What will happen if even after getting more information, agent is still confused enough to make a decision?

In [26]:
unresolved_cases = []

for transaction, evidence, belief, costs in question_transactions:

    questions = get_questions(evidence)

    if not questions:
        continue

    responses = simulate_customer_response(
        transaction,
        questions
    )

    # Use transaction confirmation if available
    if "transaction_confirmation" in responses:

        updated_belief = update_belief_with_customer_response(
            belief,
            responses["transaction_confirmation"]
        )

        updated_costs = calculate_action_costs(
            updated_belief
        )

        updated_action = choose_action(
            updated_costs
        )

        if updated_action == "Question":

            unresolved_cases.append({
                "transaction": transaction,
                "evidence": evidence,
                "initial_belief": belief,
                "response": responses["transaction_confirmation"],
                "updated_belief": updated_belief,
                "updated_costs": updated_costs
            })

print("Unresolved Question cases:", len(unresolved_cases))

Unresolved Question cases: 77


In [27]:
case = unresolved_cases[0]

transaction = case["transaction"]

print("Transaction:", transaction["transaction_id"])
print("Actual fraud:", transaction["fraud"])
print("Evidence:", case["evidence"])

print("\nInitial fraud belief:", case["initial_belief"])
print("Customer response:", case["response"])
print("Updated fraud belief:", case["updated_belief"])

print("\nUpdated action costs:")
for action, cost in case["updated_costs"].items():
    print(f"{action}: {cost:.4f}")

print("\nFinal action before human investigation: Question")

Transaction: FREQ_C0386
Actual fraud: 1
Evidence: {'amount_unusual': False, 'location_unusual': False, 'merchant_unusual': False, 'frequency_unusual': True, 'velocity_unusual': False}

Initial fraud belief: 0.5421293559204614
Customer response: True
Updated fraud belief: 0.11082182859747552

Updated action costs:
Approve: 11.0822
Question: 2.8866
Examine: 7.6675
Decline: 17.7836

Final action before human investigation: Question


## Human in Loop
If the agent asks for additional info and remains in the `Question` state even after receiving that information, then the a human is looped in.
This represents unresolved uncertainity rather that a direct fraud classification

In [28]:
def decide_after_customer_response(fraud_belief, customer_response):

    updated_belief = update_belief_with_customer_response(
        fraud_belief,
        customer_response
    )

    updated_costs = calculate_action_costs(
        updated_belief
    )

    new_action = choose_action(
        updated_costs
    )

    if new_action == "Question":
        new_action = "Human Investigation"

    return updated_belief, updated_costs, new_action

In [29]:
final_belief, final_costs, final_action = decide_after_customer_response(case["initial_belief"],case["response"])

print("Initial fraud belief:", case["initial_belief"])
print("Customer response:", case["response"])
print("Updated fraud belief:", final_belief)

print("\nFinal action costs:")
for action, cost in final_costs.items():
    print(f"{action}: {cost:.4f}")

print("\nFinal action:", final_action)

Initial fraud belief: 0.5421293559204614
Customer response: True
Updated fraud belief: 0.11082182859747552

Final action costs:
Approve: 11.0822
Question: 2.8866
Examine: 7.6675
Decline: 17.7836

Final action: Human Investigation


## Full Test-Set Evaluation

The agent is evaluated on the unseen test set.

For each transaction, the agent:

1. Calculates its initial fraud belief
2. Chooses an initial action
3. If the action is `Question`, requests additional information
4. Updates its belief using the customer response
5. Recalculates the action
6. Escalates to `Human Investigation` if uncertainty remains

In [30]:
week2_results = []

for _, transaction in test_df.iterrows():

    # ------------------------------------------------
    # 1. Extract evidence
    # ------------------------------------------------
    evidence = {
        "amount_unusual": transaction["amount_unusual"],
        "location_unusual": transaction["location_unusual"],
        "merchant_unusual": transaction["merchant_unusual"],
        "frequency_unusual": transaction["frequency_unusual"],
        "velocity_unusual": transaction["velocity_unusual"]
    }

    # ------------------------------------------------
    # 2. Initial belief
    # ------------------------------------------------
    initial_belief = calculate_fraud_belief(
        evidence,
        train_likelihoods
    )

    # ------------------------------------------------
    # 3. Initial action
    # ------------------------------------------------
    initial_costs = calculate_action_costs(
        initial_belief
    )

    initial_action = choose_action(
        initial_costs
    )

    # Default values
    updated_belief = initial_belief
    final_action = initial_action
    customer_response = None
    questions = []

    # ------------------------------------------------
    # 4. Ask for additional information if needed
    # ------------------------------------------------
    if initial_action == "Question":

        questions = get_questions(evidence)

        if questions:

            customer_responses = simulate_customer_response(
                transaction,
                questions
            )

            # Use transaction confirmation when available
            if "transaction_confirmation" in customer_responses:

                customer_response = customer_responses[
                    "transaction_confirmation"
                ]

                # ------------------------------------------------
                # 5. Update belief
                # ------------------------------------------------
                updated_belief = (
                    update_belief_with_customer_response(
                        initial_belief,
                        customer_response
                    )
                )

                # ------------------------------------------------
                # 6. Recalculate action
                # ------------------------------------------------
                updated_costs = calculate_action_costs(
                    updated_belief
                )

                final_action = choose_action(
                    updated_costs
                )

                # ------------------------------------------------
                # 7. Human investigation if still unresolved
                # ------------------------------------------------
                if final_action == "Question":
                    final_action = "Human Investigation"

    # ------------------------------------------------
    # 8. Store result
    # ------------------------------------------------
    week2_results.append({
        "transaction_id": transaction["transaction_id"],
        "actual_fraud": transaction["fraud"],
        "initial_belief": initial_belief,
        "updated_belief": updated_belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "customer_response": customer_response
    })

week2_results_df = pd.DataFrame(week2_results)

print("Total test transactions:", len(week2_results_df))

display(week2_results_df.head())

Total test transactions: 2400


,transaction_id,actual_fraud,initial_belief,updated_belief,initial_action,final_action,customer_response
0,T01628,0,0.000018,0.000018,Approve,Approve,None
1,T05425,0,0.000022,0.000022,Approve,Approve,None
2,T01442,0,0.000010,0.000010,Approve,Approve,None
3,T07612,0,0.000010,0.000010,Approve,Approve,None
4,T00310,0,0.000025,0.000025,Approve,Approve,None


In [31]:
print("Initial actions:")
print(week2_results_df["initial_action"].value_counts())

print("\nFinal actions:")
print(week2_results_df["final_action"].value_counts())

Initial actions:
initial_action
Approve     1931
Decline      237
Question     223
Examine        9
Name: count, dtype: int64

Final actions:
final_action
Approve                1931
Decline                 391
Human Investigation      69
Examine                   9
Name: count, dtype: int64


In [32]:
# Compare fraud outcomes before and after active sensing

print("Fraud transactions by initial action:")
print(
    pd.crosstab(
        week2_results_df["initial_action"],
        week2_results_df["actual_fraud"]
    )
)

print("\nFraud transactions by final action:")
print(
    pd.crosstab(
        week2_results_df["final_action"],
        week2_results_df["actual_fraud"]
    )
)

Fraud transactions by initial action:
actual_fraud       0    1
initial_action           
Approve         1931    0
Decline            0  237
Examine            2    7
Question          58  165

Fraud transactions by final action:
actual_fraud            0    1
final_action                  
Approve              1931    0
Decline                 4  387
Examine                 2    7
Human Investigation    54   15


In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Treat Decline as the automated fraud prediction
week2_results_df["predicted_fraud"] = (
    week2_results_df["final_action"] == "Decline"
).astype(int)

y_true = week2_results_df["actual_fraud"]
y_pred = week2_results_df["predicted_fraud"]

print("Week 2 Performance")
print("------------------")
print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred), 4))
print("Recall   :", round(recall_score(y_true, y_pred), 4))
print("F1 Score :", round(f1_score(y_true, y_pred), 4))

Week 2 Performance
------------------
Accuracy : 0.9892
Precision: 0.9898
Recall   : 0.9462
F1 Score : 0.9675


In [34]:
import random

random.seed(42)
if "transaction_confirmation" in customer_responses:

    customer_response = customer_responses["transaction_confirmation"]

    updated_belief = (update_belief_with_customer_response(initial_belief,customer_response))

    updated_costs = calculate_action_costs(updated_belief)

    final_action = choose_action(updated_costs)

    if final_action == "Question":
        final_action = "Human Investigation"

if customer_responses:

    # Use transaction confirmation if available
    if "transaction_confirmation" in customer_responses:

        customer_response = customer_responses[
            "transaction_confirmation"
        ]

        updated_belief = (
            update_belief_with_customer_response(
                updated_belief,
                customer_response
            )
        )

    # Use location confirmation if available
    if "location_confirmation" in customer_responses:

        location_response = customer_responses[
            "location_confirmation"
        ]

        updated_belief = (
            update_belief_with_customer_response(
                updated_belief,
                location_response
            )
        )

    updated_costs = calculate_action_costs(
        updated_belief
    )

    final_action = choose_action(
        updated_costs
    )

    if final_action == "Question":
        final_action = "Human Investigation"

In [35]:
print("Initial actions:")
print(week2_results_df["initial_action"].value_counts())

print("\nFinal actions:")
print(week2_results_df["final_action"].value_counts())

print("*"*40)

print("Fraud transactions by final action:")
print(
    pd.crosstab(
        week2_results_df["final_action"],
        week2_results_df["actual_fraud"]
    )
)

Initial actions:
initial_action
Approve     1931
Decline      237
Question     223
Examine        9
Name: count, dtype: int64

Final actions:
final_action
Approve                1931
Decline                 391
Human Investigation      69
Examine                   9
Name: count, dtype: int64
****************************************
Fraud transactions by final action:
actual_fraud            0    1
final_action                  
Approve              1931    0
Decline                 4  387
Examine                 2    7
Human Investigation    54   15


# Entropy and Uncertainty Analysis

Entropy measures how uncertain the agent is about the current fraud belief.

- Low entropy → high confidence
- High entropy → high uncertainty

We use entropy to measure whether additional evidence reduces uncertainty.

In [36]:
import math
def calculate_entropy(fraud_belief):
    legit_belief = 1 - fraud_belief

    entropy = 0

    if fraud_belief > 0:
        entropy -= fraud_belief * math.log2(fraud_belief)

    if legit_belief > 0:
        entropy -= legit_belief * math.log2(legit_belief)

    return entropy

week2_results_df["initial_entropy"] = (week2_results_df["initial_belief"].apply(calculate_entropy))

print(week2_results_df.groupby("initial_action")["initial_entropy"].mean().sort_values())

initial_action
Approve     0.000371
Decline     0.386118
Examine     0.958594
Question    0.983257
Name: initial_entropy, dtype: float64


In [37]:
print(week2_results_df[["initial_action", "initial_belief", "initial_entropy"]].groupby("initial_action")
    .agg(min_entropy=("initial_entropy", "min"),mean_entropy=("initial_entropy", "mean"),max_entropy=("initial_entropy", "max")))

                min_entropy  mean_entropy  max_entropy
initial_action                                        
Approve            0.000186      0.000371     0.001555
Decline            0.001743      0.386118     0.832303
Examine            0.955519      0.958594     0.983189
Question           0.944369      0.983257     0.999854


In [38]:
# Calculate entropy after customer response

week2_results_df["updated_entropy"] = (
    week2_results_df["updated_belief"]
    .apply(calculate_entropy)
)

# Entropy reduction = reduction in uncertainty
week2_results_df["entropy_reduction"] = (
    week2_results_df["initial_entropy"]
    - week2_results_df["updated_entropy"]
)

print(
    "Average entropy reduction:",
    week2_results_df.loc[
        week2_results_df["customer_response"].notna(),
        "entropy_reduction"
    ].mean()
)

Average entropy reduction: 0.6182059486386364


In [40]:
questioned_transactions = week2_results_df[
    week2_results_df["customer_response"].notna()
]

print("Transactions with customer response:",
      len(questioned_transactions))

print(
    "\nAverage initial entropy:",
    questioned_transactions["initial_entropy"].mean()
)

print(
    "Average updated entropy:",
    questioned_transactions["updated_entropy"].mean()
)



Transactions with customer response: 223

Average initial entropy: 0.9832573518127891
Average updated entropy: 0.365051403174153
